# BasketLens Analytics Dashboard
## Data Cleaning, Validation, and Feature Engineering Pipeline

**Purpose.** This notebook prepares the BasketLens retail source data for Power BI Desktop. It keeps the raw files immutable, creates clean working copies, applies repeatable quality controls, and exports only validated datasets.

**Data model.** `Transactions` is the fact table. `Products`, `Customers`, and `Calendar` are supporting dimensions. The pipeline intentionally stops at clean, auditable data; dashboard design happens in Power BI.


## 1. Project Overview

The analysis supports sales performance, category contribution, customer buying behavior, seasonal demand, and market-basket analysis. Every transformation is explicit and reproducible.

**Quality principles**

- Never edit a raw dataframe or raw CSV.
- Preserve original purchase-date text in `PurchaseDateRaw` before normalizing dates.
- Remove only exact duplicate transaction/product records; transactions can legitimately contain multiple products.
- Recalculate revenue from quantity and unit price rather than trusting imported totals.
- Block export if critical identifiers, dates, or financial controls fail validation.


## 2. Import Libraries

Only standard analytical libraries are used. Display settings keep notebook output compact and review-friendly.


In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
CLEAN_DIR = PROJECT_ROOT / "cleaned_data"

assert DATA_DIR.exists(), f"Input folder not found: {DATA_DIR}"
print(f"Project root: {PROJECT_ROOT}")


## 3. Load Datasets

Raw files are loaded once and are never modified. All work below begins with explicit `*_clean` copies.


In [ ]:
transactions = pd.read_csv(DATA_DIR / "BasketLens_Transactions.csv")
products = pd.read_csv(DATA_DIR / "BasketLens_Products.csv")
customers = pd.read_csv(DATA_DIR / "BasketLens_Customers.csv")
calendar = pd.read_csv(DATA_DIR / "BasketLens_Calendar.csv")

raw_datasets = {
    "Transactions": transactions,
    "Products": products,
    "Customers": customers,
    "Calendar": calendar,
}

pd.DataFrame(
    [{"Dataset": name, "Rows": len(frame), "Columns": frame.shape[1], "Memory KB": round(frame.memory_usage(deep=True).sum() / 1024, 1)}
     for name, frame in raw_datasets.items()]
)


## 4. Dataset Overview

This checkpoint profiles schema and sample values before transformation. It is deliberately descriptive, not a cleaning operation.


In [ ]:
schema_overview = pd.concat(
    [pd.DataFrame({"Dataset": name, "Column": frame.columns, "Raw dtype": frame.dtypes.astype(str).values})
     for name, frame in raw_datasets.items()],
    ignore_index=True,
)
display(schema_overview)
display(transactions.head(3))


## 5. Exploratory Data Analysis

Small frequency tables surface unexpected category labels and guide standardization rules without flooding the notebook with raw rows.


In [ ]:
def frequency_table(frame: pd.DataFrame, column: str, top_n: int = 10) -> pd.DataFrame:
    """Return a compact count and share table, including missing values."""
    counts = frame[column].fillna("<MISSING>").astype(str).value_counts().head(top_n)
    return pd.DataFrame({
        "Value": counts.index,
        "Count": counts.values,
        "Share %": (counts.values / len(frame) * 100).round(2),
    })

eda_summary = {
    "Product categories": frequency_table(transactions, "ProductCategory"),
    "Store locations": frequency_table(transactions, "StoreLocation"),
    "Customer segments": frequency_table(customers, "CustomerSegment"),
}
for title, table in eda_summary.items():
    print(f"\n{title}")
    display(table)


## 6. Data Quality Assessment

Before cleaning, quantify missingness, exact duplicates, duplicate business keys, and malformed dates. This baseline makes the effect of every corrective action auditable.


In [ ]:
def quality_checkpoint(datasets: dict[str, pd.DataFrame], stage: str) -> pd.DataFrame:
    """Summarize shape, missing cells, exact duplicates, and dtype readiness at a named stage."""
    rows = []
    for name, frame in datasets.items():
        rows.append({
            "Stage": stage,
            "Dataset": name,
            "Rows": len(frame),
            "Columns": frame.shape[1],
            "Missing cells": int(frame.isna().sum().sum()),
            "Exact duplicate rows": int(frame.duplicated().sum()),
        })
    return pd.DataFrame(rows)

def missing_value_table(datasets: dict[str, pd.DataFrame]) -> pd.DataFrame:
    """Return only columns with missing values, including missing percentage."""
    records = []
    for name, frame in datasets.items():
        for column, count in frame.isna().sum().items():
            if count:
                records.append({"Dataset": name, "Column": column, "Missing": int(count), "Missing %": round(count / len(frame) * 100, 2)})
    return pd.DataFrame(records)

raw_checkpoint = quality_checkpoint(raw_datasets, "Raw baseline")
display(raw_checkpoint)
display(missing_value_table(raw_datasets))

duplicate_key_assessment = pd.DataFrame({
    "Rule": ["TransactionID + ProductID", "ProductID", "CustomerID", "Calendar Date"],
    "Duplicate records": [
        int(transactions.duplicated(["TransactionID", "ProductID"]).sum()),
        int(products.duplicated(["ProductID"]).sum()),
        int(customers.duplicated(["CustomerID"]).sum()),
        int(calendar.duplicated(["Date"]).sum()),
    ],
})
duplicate_key_assessment


## 7. Data Cleaning

The functions below are intentionally modular. They clean a copy and return it, making the process safe to rerun if a dataframe must be recreated.

**Date convention:** ISO dates remain ISO; slash dates are interpreted as month/day/year; non-ISO hyphen dates are interpreted as day-month-year. The raw text is retained for auditability.


In [ ]:
def clean_column_names(frame: pd.DataFrame) -> pd.DataFrame:
    """Strip whitespace and punctuation from headers while preserving readable PascalCase names."""
    result = frame.copy()
    result.columns = [re.sub(r"[^A-Za-z0-9]+", "", str(column).strip()) for column in result.columns]
    return result

def trim_text(frame: pd.DataFrame) -> pd.DataFrame:
    """Remove leading/trailing whitespace and invisible control characters from text fields."""
    result = frame.copy()
    for column in result.select_dtypes(include=["object", "string"]).columns:
        result[column] = (result[column].astype("string")
                          .str.replace(r"[\r\n\t\x00-\x1F\x7F-\x9F]", "", regex=True)
                          .str.strip())
    return result

def standardize_text(frame: pd.DataFrame, columns: list[str], uppercase: bool = False) -> pd.DataFrame:
    """Apply consistent title case to labels or upper case to identifier fields."""
    result = frame.copy()
    for column in columns:
        if column in result.columns:
            values = result[column].astype("string")
            result[column] = values.str.upper() if uppercase else values.str.title()
    return result

def clean_currency(frame: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    """Remove currency symbols and separators, then coerce monetary fields to numeric values."""
    result = frame.copy()
    for column in columns:
        if column in result.columns:
            result[column] = pd.to_numeric(result[column].astype("string").str.replace(r"[$,]", "", regex=True), errors="coerce")
    return result

def parse_dates(frame: pd.DataFrame, column: str) -> pd.DataFrame:
    """Parse ISO, MDY slash, and DMY hyphen dates without discarding the source text."""
    result = frame.copy()
    raw_column = f"{column}Raw"
    result[raw_column] = result[column].astype("string")
    values = result[raw_column].str.strip()
    parsed = pd.Series(pd.NaT, index=result.index, dtype="datetime64[ns]")
    formats = [
        (r"^\d{4}-\d{2}-\d{2}$", "%Y-%m-%d"),
        (r"^\d{1,2}/\d{1,2}/\d{4}$", "%m/%d/%Y"),
        (r"^\d{1,2}-\d{1,2}-\d{4}$", "%d-%m-%Y"),
    ]
    for pattern, date_format in formats:
        mask = values.str.match(pattern, na=False)
        parsed.loc[mask] = pd.to_datetime(values.loc[mask], format=date_format, errors="coerce")
    unresolved = parsed.isna() & values.notna() & values.ne("")
    parsed.loc[unresolved] = pd.to_datetime(values.loc[unresolved], errors="coerce")
    result[column] = parsed
    return result

def handle_missing_values(transactions_frame: pd.DataFrame, customers_frame: pd.DataFrame, calendar_frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Apply documented business-safe imputations without silently dropping records."""
    tx, cust, cal = transactions_frame.copy(), customers_frame.copy(), calendar_frame.copy()
    quantity = pd.to_numeric(tx["Quantity"], errors="coerce").astype(float)
    quantity = quantity.fillna(quantity.groupby(tx["ProductID"]).transform("median")).fillna(quantity.median())
    tx["Quantity"] = quantity.round().clip(lower=1).astype("Int64")
    cust["LoyaltyTier"] = cust["LoyaltyTier"].fillna("Not Enrolled")
    cal["Holiday"] = cal["Holiday"].fillna("None")
    return tx, cust, cal

def convert_data_types(frame: pd.DataFrame, integer_columns: list[str] | None = None) -> pd.DataFrame:
    """Apply numeric types after text and currency cleanup."""
    result = frame.copy()
    for column in integer_columns or []:
        if column in result.columns:
            result[column] = pd.to_numeric(result[column], errors="coerce").astype("Int64")
    return result


In [ ]:
# Create clean copies. The raw dataframes above remain untouched.
transactions_clean = transactions.copy()
products_clean = products.copy()
customers_clean = customers.copy()
calendar_clean = calendar.copy()

# Header and text sanitation.
transactions_clean = standardize_text(trim_text(clean_column_names(transactions_clean)), ["TransactionID", "CustomerID", "ProductID"], uppercase=True)
products_clean = standardize_text(trim_text(clean_column_names(products_clean)), ["ProductID"], uppercase=True)
customers_clean = standardize_text(trim_text(clean_column_names(customers_clean)), ["CustomerID"], uppercase=True)
calendar_clean = trim_text(clean_column_names(calendar_clean))

transactions_clean = standardize_text(transactions_clean, ["ProductName", "ProductCategory", "CustomerSegment", "StoreLocation", "PaymentType"])
products_clean = standardize_text(products_clean, ["ProductName", "Category", "Department"])
customers_clean = standardize_text(customers_clean, ["CustomerSegment", "AgeGroup", "LoyaltyTier", "Region"])

# Type conversion, safe date parsing, and business-rule missing value treatment.
transactions_clean = clean_currency(transactions_clean, ["UnitPrice", "TotalAmount"])
products_clean = clean_currency(products_clean, ["UnitPrice"])
transactions_clean = convert_data_types(transactions_clean, ["Quantity"])
transactions_clean = parse_dates(transactions_clean, "PurchaseDate")
calendar_clean = parse_dates(calendar_clean, "Date")
transactions_clean, customers_clean, calendar_clean = handle_missing_values(transactions_clean, customers_clean, calendar_clean)

clean_checkpoint_before_dedup = quality_checkpoint({"Transactions": transactions_clean, "Products": products_clean, "Customers": customers_clean, "Calendar": calendar_clean}, "After cleaning before deduplication")
display(clean_checkpoint_before_dedup)


## 8. Data Validation

Validation is a release gate, not a visual check. The controls below identify invalid IDs, invalid dates, negative values, duplicate business keys, unmatched dimensions, and inconsistent imported totals. The imported total is retained only as an audit comparison before it is replaced.


In [ ]:
def validation_table(transactions_frame: pd.DataFrame, products_frame: pd.DataFrame, customers_frame: pd.DataFrame, calendar_frame: pd.DataFrame) -> pd.DataFrame:
    """Return a clear pass/fail control table for final dataset readiness."""
    imported_total = transactions_frame["TotalAmount"].copy()
    expected_total = (transactions_frame["Quantity"].astype(float) * transactions_frame["UnitPrice"]).round(2)
    checks = [
        ("Missing transaction identifiers", transactions_frame["TransactionID"].isna().sum() + transactions_frame["TransactionID"].eq("").sum()),
        ("Missing customer identifiers", transactions_frame["CustomerID"].isna().sum() + transactions_frame["CustomerID"].eq("").sum()),
        ("Missing product identifiers", transactions_frame["ProductID"].isna().sum() + transactions_frame["ProductID"].eq("").sum()),
        ("Invalid purchase dates", transactions_frame["PurchaseDate"].isna().sum()),
        ("Negative or zero quantities", transactions_frame["Quantity"].le(0).sum()),
        ("Negative or zero unit prices", transactions_frame["UnitPrice"].le(0).sum()),
        ("Duplicate TransactionID/ProductID pairs", transactions_frame.duplicated(["TransactionID", "ProductID"]).sum()),
        ("Duplicate product dimension keys", products_frame.duplicated(["ProductID"]).sum()),
        ("Duplicate customer dimension keys", customers_frame.duplicated(["CustomerID"]).sum()),
        ("Duplicate calendar dates", calendar_frame.duplicated(["Date"]).sum()),
        ("Unmatched product IDs", (~transactions_frame["ProductID"].isin(products_frame["ProductID"])).sum()),
        ("Unmatched customer IDs", (~transactions_frame["CustomerID"].isin(customers_frame["CustomerID"])).sum()),
        ("Imported TotalAmount differs from Quantity x UnitPrice", (~np.isclose(imported_total.fillna(-1), expected_total.fillna(-1))).sum()),
    ]
    result = pd.DataFrame(checks, columns=["Control", "Issues found"])
    result["Status"] = np.where(result["Issues found"].eq(0), "PASS", "REVIEW")
    return result

validation_before_dedup = validation_table(transactions_clean, products_clean, customers_clean, calendar_clean)
display(validation_before_dedup)


## 9. Feature Engineering

Duplicate removal is now applied at the approved business grain: a repeated `TransactionID` plus `ProductID` record is redundant, while different products in the same transaction are retained. Canonical labels are restored from dimension tables, and all sales measures are recalculated.


In [ ]:
# Remove only redundant transaction-product rows and exact dimension duplicates.
transactions_clean = transactions_clean.drop_duplicates(["TransactionID", "ProductID"], keep="first").reset_index(drop=True)
products_clean = products_clean.drop_duplicates(["ProductID"], keep="first").reset_index(drop=True)
customers_clean = customers_clean.drop_duplicates(["CustomerID"], keep="first").reset_index(drop=True)
calendar_clean = calendar_clean.drop_duplicates(["Date"], keep="first").reset_index(drop=True)

# Canonical dimension values prevent inconsistent transaction labels from reaching Power BI.
transactions_clean = transactions_clean.drop(columns=["ProductName", "ProductCategory", "CustomerSegment"], errors="ignore")
transactions_clean = transactions_clean.merge(products_clean[["ProductID", "ProductName", "Category", "Department"]], on="ProductID", how="left", validate="many_to_one")
transactions_clean = transactions_clean.merge(customers_clean[["CustomerID", "CustomerSegment", "AgeGroup", "LoyaltyTier", "Region"]], on="CustomerID", how="left", validate="many_to_one")

# Recalculate the authoritative financial metric and useful Power BI fields.
transactions_clean["TotalAmount"] = (transactions_clean["Quantity"].astype(float) * transactions_clean["UnitPrice"]).round(2)
transactions_clean["BasketItemCount"] = transactions_clean.groupby("TransactionID")["ProductID"].transform("size").astype("Int64")
transactions_clean["BasketValue"] = transactions_clean.groupby("TransactionID")["TotalAmount"].transform("sum").round(2)
transactions_clean["BasketSizeSegment"] = pd.cut(transactions_clean["BasketItemCount"], [0, 1, 3, np.inf], labels=["Single", "Small", "Large"]).astype("string")
products_clean["PriceTier"] = pd.cut(products_clean["UnitPrice"], [-np.inf, 2, 5, np.inf], labels=["Low", "Medium", "Premium"]).astype("string")
customers_clean["IsLoyal"] = customers_clean["LoyaltyTier"].ne("Not Enrolled").astype("Int64")
calendar_clean["IsHoliday"] = calendar_clean["Holiday"].ne("None").astype("Int64")

# Use stable ISO strings in the exported CSV files.
transactions_clean["PurchaseDate"] = transactions_clean["PurchaseDate"].dt.strftime("%Y-%m-%d")
calendar_clean["Date"] = calendar_clean["Date"].dt.strftime("%Y-%m-%d")

final_checkpoint = quality_checkpoint({"Transactions": transactions_clean, "Products": products_clean, "Customers": customers_clean, "Calendar": calendar_clean}, "Final engineered data")
display(final_checkpoint)


## 10. Export Cleaned Data

The final validation must pass before export. This prevents incomplete or unreliable files from entering Power BI. Exports are written to `cleaned_data/`; raw files remain unchanged.


In [ ]:
# Re-run validation after duplicate removal and feature engineering.
transactions_for_validation = transactions_clean.copy()
transactions_for_validation["PurchaseDate"] = pd.to_datetime(transactions_for_validation["PurchaseDate"], errors="coerce")
calendar_for_validation = calendar_clean.copy()
calendar_for_validation["Date"] = pd.to_datetime(calendar_for_validation["Date"], errors="coerce")
final_validation = validation_table(transactions_for_validation, products_clean, customers_clean, calendar_for_validation)
display(final_validation)

blocking_controls = final_validation.loc[final_validation["Control"].ne("Imported TotalAmount differs from Quantity x UnitPrice"), "Issues found"]
assert blocking_controls.eq(0).all(), "Export blocked: resolve failing data-quality controls above."

CLEAN_DIR.mkdir(parents=True, exist_ok=True)
transactions_clean.to_csv(CLEAN_DIR / "Transactions_Clean.csv", index=False)
products_clean.to_csv(CLEAN_DIR / "Products_Clean.csv", index=False)
customers_clean.to_csv(CLEAN_DIR / "Customers_Clean.csv", index=False)
calendar_clean.to_csv(CLEAN_DIR / "Calendar_Clean.csv", index=False)

export_summary = pd.DataFrame([
    {"File": "Transactions_Clean.csv", "Rows": len(transactions_clean), "Columns": transactions_clean.shape[1]},
    {"File": "Products_Clean.csv", "Rows": len(products_clean), "Columns": products_clean.shape[1]},
    {"File": "Customers_Clean.csv", "Rows": len(customers_clean), "Columns": customers_clean.shape[1]},
    {"File": "Calendar_Clean.csv", "Rows": len(calendar_clean), "Columns": calendar_clean.shape[1]},
])
display(export_summary)
